### Dataset and Task Metadata

In [41]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="brasilian_houses",
    dataset_year="2020",
    domain_str="finance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.openml.org/data/download/22044541/dataset",
    download_description="""
We download the data from OpenML.

In this notebook, run:

    import openml

    # Load the dataset object from OpenML
    dataset = openml.datasets.get_dataset(
        42688,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
        )
    dataset.get_data()[0].to_csv("../../../local-data-warehouse/brasilian_houses/brasilian_houses.csv", index=False)
""",
    # References
    academic_reference_bibtex=r"""@misc{farias2020brasilian,
    author = {Fernando Lucas de Oliveira Farias and Raphael Fontes},
    title = {Ensino 5A > Datacamp},
    year = {2020},
    howpublished = {\url{https://kaggle.com/competitions/ensino-5a-datacamp}},
    note = {Kaggle}
}
""",
    academic_reference_bibtex_key="farias2020brasilian",
    license="Kaggle Competition Rules",
    data_tags=["IID"],
    curation_comments="""
 - We start with the data from Kaggle and try to get the data from the competition, but it is a limited-participation competition. Only invited users may participate. So instead, we searched for other sources and found https://www.kaggle.com/datasets/antonyroy/finaltry/data and https://www.openml.org/d/42688, which seem to be same data, since they are citing this competition, so we use the OpenML data.
 
 - We mapped binary columns to 0 and 1 and checked if the data is IID
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="total_(BRL)",
    problem_type="regression",
    objective_metric_name="rmse",
    stratify_on="total_(BRL)",
)

## Preprocessing

In [55]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/brasilian_houses.csv", header=0)

feature_names = [
    "city",
    "area",
    "rooms",
    "bathroom",
    "parking_spaces",
    "floor",
    "animal",
    "furniture",
    "hoa_(BRL)",
    "rent_amount_(BRL)",
    "property_tax_(BRL)",
    "fire_insurance_(BRL)",
    "total_(BRL)"
]

df.columns = feature_names
df.rename(columns={"animal": "animal_accepted"}, inplace=True)
df.rename(columns={"furniture": "furnished"}, inplace=True)

cat_features = [
    "city",
    "area",
    "animal_accepted",
    "furnished",
]

df["animal_accepted"] = df["animal_accepted"].map({"acept": "Yes", "not acept": "No"}).astype("category")
df["furnished"] = df["furnished"].map({"not furnished": "No", "furnished": "Yes"}).astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df[cat_features] = df[cat_features].astype("category")

In [56]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,city,area,rooms,bathroom,parking_spaces,floor,animal_accepted,furnished,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
0,Campinas,134,3,3,2,7,No,No,1000,2500,177,32,3709
1,Rio de Janeiro,80,2,1,1,10,No,No,660,1900,50,25,2635
2,Campinas,80,2,2,1,8,Yes,Yes,860,2000,34,26,2920
3,Belo Horizonte,90,3,2,1,2,Yes,No,200,1400,111,19,1730
4,Sao Paulo,300,4,2,0,5,Yes,No,6000,15000,834,191,22030


## Data Checks

In [57]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 10,692
Columns: 13
Use sampling: False (sample size: 10,692)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['hoa_(BRL)', 'property_tax_(BRL)', 'rent_amount_(BRL)', 'area', 'fire_insurance_(BRL)', 'floor', 'rooms', 'parking_spaces', 'bathroom', 'city']
Rows remaining as candidates after top-10 filter: 625 (of 10,692)

#### Duplicate Report
Total duplicate rows: 358 (3.35% of dataset)
Duplicate rows ignoring target: 358 (3.35% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [58]:
# Sample Rows
df_head

,city,area,rooms,bathroom,parking_spaces,floor,animal_accepted,furnished,hoa_(BRL),rent_amount_(BRL),property_tax_(BRL),fire_insurance_(BRL),total_(BRL)
0,Campinas,134,3,3,2,7,No,No,1000,2500,177,32,3709
1,Rio de Janeiro,80,2,1,1,10,No,No,660,1900,50,25,2635
2,Campinas,80,2,2,1,8,Yes,Yes,860,2000,34,26,2920
3,Belo Horizonte,90,3,2,1,2,Yes,No,200,1400,111,19,1730
4,Sao Paulo,300,4,2,0,5,Yes,No,6000,15000,834,191,22030


In [59]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,city,category,0.0,0.0,5.0,"Sao Paulo, Rio de Janeiro, Belo Horizonte, Porto Alegre, Campinas"
1,area,category,0.0,0.0,517.0,"50, 70, 60, 100, 80, 40, 90, 200, 45, 120"
2,animal_accepted,category,0.0,0.0,2.0,"Yes, No"
3,furnished,category,0.0,0.0,2.0,"No, Yes"
4,rooms,int64,0.0,0.0,11.0,"3, 2, 1, 4, 5, 6, 7, 8, 10, 13"
5,bathroom,int64,0.0,0.0,10.0,"1, 2, 3, 4, 5, 6, 7, 8, 9, 10"
6,parking_spaces,int64,0.0,0.0,11.0,"1, 0, 2, 3, 4, 5, 6, 8, 7, 10"
7,floor,int64,0.0,0.0,35.0,"0, 1, 2, 3, 4, 5, 6, 7, 8, 9"
8,hoa_(BRL),int64,0.0,0.0,1679.0,"0, 400, 300, 500, 600, 450, 350, 700, 1000, 2000"
9,rent_amount_(BRL),int64,0.0,0.0,1195.0,"2500, 2000, 1200, 3000, 15000, 3500, 1800, 1500, 4000, 1100"


In [50]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
rooms,10692.0,2.506079,1.171266,1.0,13.0
bathroom,10692.0,2.236813,1.407198,1.0,10.0
parking_spaces,10692.0,1.609147,1.589521,0.0,12.0
floor,10692.0,5.067995,6.069050,0.0,301.0
hoa_(BRL),10692.0,1174.021698,15592.305248,0.0,1117000.0
rent_amount_(BRL),10692.0,3896.247194,3408.545518,450.0,45000.0
property_tax_(BRL),10692.0,366.704358,3107.832321,0.0,313700.0
fire_insurance_(BRL),10692.0,53.300879,47.768031,3.0,677.0
total_(BRL),10692.0,5490.487000,16484.725912,499.0,1120000.0


In [51]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                              
animal_accepted 1                Yes   8316  77.78
                2                 No   2376  22.22
area            1                 50    334   3.12
                2                 70    329   3.08
                3                 60    297   2.78
                4                100    253   2.37
                5                 80    253   2.37
city            1          Sao Paulo   5887  55.06
                2     Rio de Janeiro   1501  14.04
                3     Belo Horizonte   1258  11.77
                4       Porto Alegre   1193  11.16
                5           Campinas    853   7.98
furnished       1                 No   8086  75.63
                2                Yes   2606  24.37

In [52]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,58.961,0.298,2.717462e+08,0.638,log,205619.0,202624.8,lognormal


## Task Curation

In [53]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [54]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/sklearn/model_selection/_split.py:784: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_of_target_y = type_of_target(y)
/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(
/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/sklearn/model_selection/_split.py:784: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_of_target_y = type_of_target(y)
/Users/schaefer.bastian/miniconda3/envs/jupyter_env-py3-10/lib/python3.10/site-packages/sklearn/model_selecti

## Export

In [40]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d1a3e-cef0-7601-8b04-7abd548bf020
303b1bf7f656e5f5f2fb8d55810a35abd2539edcb35e7fac431335c9b9d7c06a
